# 18 - Does GREEDY DECODE produce better patches?

**Two arms, `v10_clamp_timecap` (control, `temperature: 0.2`) then `v11_greedy`
(`temperature: 0.0`), over the 60 fastapi tasks of `clean_dev`, at
`max_tool_calls=100` and `max_time_minutes=30` -- the cap both bundles ship in
their own `eval_config.yaml`.** Both arms in **one session**. Pairing across
sessions is never an option here: identical v00 bytes read `gold_hit` 36/60 and
29/60 in two sessions, and one cross-session pairing read **+0.167 where the
truth was 0.000**.

## Why this run exists

Nine bundles have failed to beat the competition's own published baseline. Every
one of them moved navigation, budget, topology or `thinking_budget`.
**`temperature: 0.2 / top_p: 0.95 / top_k: 40` is byte-identical in all ten
bundles built so far** (verified 2026-10-21). Decode sampling is the one knob
class never varied, and it is one schema-legal line.

The mechanism is about error tolerance, not intelligence. Prose tolerates a
suboptimal token; **a diff does not.** An `edit_file` argument has to name a real
identifier and reproduce its context exactly, so one plausible-but-wrong token
inside a hunk means the patch fails to apply, or applies and fails its test.
`resolved` is binary per task. At temperature 0.2 the model samples on every
token, including those. Greedy decode takes the argmax instead.

## The arms differ by ONE line

| | `v10_clamp_timecap` | `v11_greedy` |
|---|---|---|
| `temperature` | **0.2** | **0.0** |
| `top_p` / `top_k` | 0.95 / 40 | 0.95 / 40 (inert at t=0, deliberately untouched) |
| `thinking_budget` | 1024 | 1024 |
| `eval_config.yaml` | `max_time_minutes: 30` | byte-identical |
| prompts, topology, tools | | byte-identical |
| `bundle_hash` | `9033b8826a9add43` | `422e1a0746a14400` |

`diff -r` reports two differing files: `MANIFEST.yaml` (lab metadata, never
shipped) and `configs/sampling.yaml`, whose only non-comment difference is the
`temperature:` line.

## `resolved` DECIDES. `gold_hit` DOES NOT.

`gold_hit` is **retired as a ranking metric** and appears here as a descriptive
column only. Same-byte sessions swung it **0.117** -- larger than the effect it
was being used to detect. A `gold_hit` gain is not a reason to ship v11 and a
`gold_hit` loss is not a reason to kill it. `stats.decide` keys on `resolved`.

## Power, stated before the run, and it is unflattering

`se_unit` is ~0.030 on 129 tasks, so ~0.044 on 60, and `detectable_delta` at
**k=1 per arm is 0.093** -- about 6 of 60 tasks. Against the 0.05 design target
`stats.is_underpowered` returns True, so the honest verdict space is
**UNDERPOWERED** or an effect large enough to be surprising.

The base rate is the harder problem, not n. P007 saw `resolved` 3/60 vs 3/60
with **zero discordant pairs**. k=2 at this shape is ~11.6 h of wall clock, past
the 9-12 h session cap; k=2 on 36 tasks detects 0.085, no better, for more
quota. **So no affordable local design adjudicates `resolved` at this base
rate**, and that is said here rather than discovered afterwards.

What this run can genuinely read is the **guard-rail**, below -- which is where
greedy decode would do its damage.

## The guard-rail, and the pre-registered kill

Greedy decode is deterministic given the context, so an agent that re-reads the
same file and re-issues the same failing edit can get **the same tokens back**.
The validator warns about exactly this: *"temperature 0 can trap the agent
re-issuing one failing edit forever; because it keeps calling tools the 3-nudge
rule never fires and the budget drains."* The pathology is measured here, not
hypothetical: `reissue_rate` rose **+0.151** when a budget was freed, with
**174** `submit_patch` calls in one task and **136** malformed `edit_file` in
another.

So this notebook computes `reissue_rate`, `coder_call_share`, `counted_calls`
and `workspace_writes` per arm **on Kaggle, into the log**, not only into the
results zip -- a guard-rail that arrives only when the run went well is not one.

**KILL CRITERION: `d_reissue_rate >= +0.100` paired, or any task in the v11 arm
with a pathological repeat count that has no counterpart in v10, refutes the
knob.** Do not then retune to 0.1: what is refuted is determinism, not the size
of 0.2.

## Pre-registered readings

Full version in `bundles/v11_greedy/MANIFEST.yaml`, written before any result
existed. In short:

- **`resolved` up significantly** -> the first intervention in eleven bundles to
  move the ship metric. Check the discordant-pair count by hand first; this
  outcome requires a large effect given the power above.
- **`resolved` flat, UNDERPOWERED** -> **the expected outcome.** The knob is not
  refuted, it is unmeasured. Record it as UNDERPOWERED, never as HOLD. Then
  decide on the guard-rail: reissue flat-or-down and timeouts flat-or-down means
  greedy is free, and a free change to the one untried knob class is worth a
  submission slot.
- **reissue rate up materially** -> killed, see above.
- **`resolved` down significantly** -> sampling was load-bearing; the jitter was
  buying escape from local minima. That is a real result and worth the quota: it
  closes the last untried knob class.
- **timeouts up** -> read with the reissue rate. Greedy changes which tokens are
  produced, not how many, so a timeout rise is most likely the loop.

## Read nothing else off this run

- **Not the leaderboard.** The private set is 58 tasks, one task is 0.0172, and
  identical bytes have scored 0.12 / 0.13 / 0.15.
- **Not a time series** against P005-P009. Different cap, different task set,
  different control arm -- the control here is **v10**, not v00.
- **Count the rows before reading anything: expect 120** (60 x 2). P009 v1
  returned 0 rows and still reported COMPLETE.

## Cost: ~10.5-13 GPU-h, from a MEASURED run of this exact shape

**P008 ran 60 tasks, 2 arms, concurrency 12, a 30-minute cap -- identical to
this -- and cost 10.57 GPU-h.** That measurement is the estimate, not an
arithmetic reconstruction of it. The arithmetic is shown only as a check: 5
waves x 30 min = 150 min of agent time per arm, plus ~25 min for install and the
26 GB model load, is ~175 min per arm and ~5.8 h of wall for two, which at the
measured ~2 GPU-h per wall hour on L4 x4 gives ~11.7 -- consistent with 10.57.

Plan against **13 GPU-h**, for two reasons that are specific rather than
cautious. P008's estimate of ~6 GPU-h came from pricing waves and quoting them
as GPU-hours, a 1.76x underrun; and P009 saw the competition shard take 1,059 s
to read from `/kaggle/input` against 89.78 s on 2026-10-01, which is load time
nobody can predict. Against it: **both arms here carry the clamp**
(`thinking_budget: 1024`), where P008's control arm was the slow 4096 one, so
this should run no longer than P008 did.

`kaggle quota` read **27.78 h remaining, refreshing 2026-10-24** at 2026-10-21
14:0x UTC -- but a cap60 run was still RUNNING at that moment and had not
finished drawing. **Re-read `kaggle quota` before pushing**; if the margin is
under ~15 h, wait for the 10-24 refresh rather than running a half-sized version.
Well inside the 9-12 h session cap either way.

Each arm's ledger rows are echoed the moment that arm finishes, so a session
lost during arm 2 still banks arm 1.

In [ ]:
import glob
import json
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

NL = chr(10)
LOG = Path('/kaggle/working/smoke_log.txt')
LOG.parent.mkdir(parents=True, exist_ok=True)


def say(msg=''):
    line = '[' + time.strftime('%H:%M:%S') + '] ' + str(msg)
    print(line, flush=True)
    with LOG.open('a', encoding='utf-8') as fh:
        fh.write(line + NL)


def nvsmi(fields):
    r = subprocess.run(['nvidia-smi', '--query-gpu=' + fields, '--format=csv,noheader'],
                       capture_output=True, text=True)
    return r.stdout.strip()


say('=== GPU ===')
info = nvsmi('name,memory.total,compute_cap')
say(info or '(nvidia-smi returned nothing)')
say('CPUs: ' + str(os.cpu_count()))

rows = [r.split(',') for r in info.splitlines() if r.strip()]
N_GPU = len(rows)
MEM_GB = sum(float(r[1].strip().split()[0]) for r in rows) / 1024 if rows else 0.0
caps = []
for r in rows:
    if len(r) > 2:
        try:
            caps.append(float(r[2].strip()))
        except ValueError:
            pass
CC = min(caps) if caps else 0.0
say('GPUs ' + str(N_GPU) + '  total ' + format(MEM_GB, '.1f') + ' GB  compute cap ' + str(CC))

assert N_GPU >= 2, 'need at least 2 GPUs, got ' + str(N_GPU)

# W4A16 compressed-tensors uses Marlin kernels, which normally need Ampere (8.0+).
# T4 is Turing (7.5). Warn and continue: whether vLLM refuses is what we are testing.
if CC and CC < 8.0:
    say('WARNING compute capability ' + str(CC) + ' is below 8.0; W4A16 Marlin')
    say('        kernels normally need Ampere or newer. vLLM may refuse.')

# 1.21 MB/token measured from the 4xL4 run; weights are ~17 GB, sharded across cards.
KV_GB = MEM_GB * 0.90 - 17.0
FITS = int(KV_GB * 1024 / 1.21) if KV_GB > 0 else 0
MAX_MODEL_LEN = 32768 if FITS >= 32768 else max(4096, (FITS // 2048) * 2048)
say('KV budget ' + format(KV_GB, '.1f') + ' GB -> about ' + format(FITS, ',') + ' tokens')
say('using max_model_len=' + str(MAX_MODEL_LEN))
assert KV_GB > 2, 'only ' + format(KV_GB, '.1f') + ' GB for KV after weights; cannot serve'
if MAX_MODEL_LEN != 32768:
    say('NOTE below the grader 32768 -> PIPELINE TEST, not a comparable measurement')

In [ ]:
# Hard gate on the accelerator, before anything expensive runs.
#
# `enable_gpu` is a boolean; the GPU TYPE comes from `machine_shape` in
# kernel-metadata.json, which an older CLI accepts and ignores without a word.
# So the notebook verifies what it was actually given rather than what it asked
# for. Weights plus overhead are ~26 GB across 4 cards and the grader's 32,768
# context needs ~39 GB more of KV at 1.21 MB/token: only L4 x4 (92 GB) holds it.
_want = 'L4'
_names = [r.split(',')[0].strip() for r in nvsmi('name').splitlines() if r.strip()]
say('accelerator: ' + str(_names))
assert len(_names) >= 4, (
    'expected 4 GPUs, got ' + str(len(_names)) + ' -- machine_shape was not honoured. '
    'Check "machine_shape": "NvidiaL4" in kernel-metadata.json and that the Kaggle '
    'CLI is >= 2.2; an invalid value is silently ignored.')
assert all(_want.lower() in n.lower() for n in _names), (
    'expected ' + _want + ' x4, got ' + str(_names) + '. A T4 x2 allocation has '
    'already cost this project a run: it cannot load the weights at all.')
assert MEM_GB > 80, (
    'only ' + format(MEM_GB, '.1f') + ' GB of VRAM; the weights plus a 32k context '
    'need ~65 GB and this run would die after the install.')
assert MAX_MODEL_LEN == 32768, (
    'MAX_MODEL_LEN is ' + str(MAX_MODEL_LEN) + ', not the grader 32768. The KV budget '
    'computed from this GPU set cannot hold the grader context, so any result would '
    'be a pipeline test rather than a measurement.')
say('accelerator gate OK: ' + str(len(_names)) + ' x ' + _want
    + ', ' + format(MEM_GB, '.1f') + ' GB, max_model_len ' + str(MAX_MODEL_LEN))

In [ ]:
for k, v in {
    'LITELLM_LOCAL_MODEL_COST_MAP': 'True',
    'TRANSFORMERS_NO_TF': '1',
    'VLLM_WORKER_MULTIPROC_METHOD': 'spawn',
    'VLLM_ENGINE_READY_TIMEOUT_S': '1200',
    'VLLM_NO_USAGE_STATS': '1',
    'OTEL_SDK_DISABLED': 'true',
    'PYTORCH_CUDA_ALLOC_CONF': 'expandable_segments:True',
}.items():
    os.environ[k] = v

INPUT = Path('/kaggle/input')
say('attached: ' + ', '.join(p.name for p in sorted(INPUT.iterdir())))


def find_dir(needles, must_contain=''):
    hits = []
    for p in INPUT.rglob('*'):
        if not p.is_dir():
            continue
        if any(n in p.name.lower() for n in needles):
            if not must_contain or any(p.glob(must_contain)):
                hits.append(p)
    return sorted(hits, key=lambda q: len(str(q)))[0] if hits else None


def find_model_dir(slug):
    # Kaggle mounts models at .../<instance>/<version>; matching the instance name
    # lands one level above the weights and vLLM rejects it. Anchor on config.json.
    hits = [q.parent for q in INPUT.rglob('config.json') if slug in str(q.parent).lower()]
    return sorted(hits, key=lambda q: len(str(q)))[0] if hits else None


WHEELHOUSE = find_dir(['wheelhouse'], '*.whl')
LAB = find_dir(['swegemma-lab'])
DATA = Path('/kaggle/input/competitions/gemma-4-developer-agent')
MODEL = find_model_dir('gemma-4-31b-it-qat-w4a16-ct')
for name, val in (('wheelhouse', WHEELHOUSE), ('lab', LAB), ('data', DATA), ('model', MODEL)):
    say('  ' + name.ljust(11) + str(val))
assert WHEELHOUSE and LAB and DATA.is_dir() and MODEL, 'missing an attached source'
assert (MODEL / 'config.json').is_file(), 'no config.json in ' + str(MODEL)

In [ ]:
# Decide whether this image can run the wheelhouse AT ALL, and if so with what.
#
# History, because the shape of this check changed twice:
#   - The wheelhouse ships cp312 and no cp313 wheel.
#   - Kaggle moved the GPU image to Python 3.13 on 2026-10-07, in both batch and
#     interactive mode. Pinning does not help: docker_image_pinning_type=original
#     resolves to today's image (verified -- KAGGLE_DOCKER_IMAGE matched the
#     pinned digest exactly).
#   - An earlier version of this cell asserted "interpreter tag must appear in the
#     wheelhouse" and failed the run. That was TOO BLUNT. Measured on the 3.13
#     image: vllm ships cp38-abi3 so it installs anywhere, torch is not in the
#     wheelhouse at all (it comes from the image, 2.11.0+cu128), and only 9 of 41
#     wheels are genuinely version-locked. With those 9 skipped, `import vllm`
#     succeeded and torch.cuda.is_available() was True.
#
# So the real question is not "do the tags match" but "can every version-locked
# wheel be satisfied". We vendor cp313 builds of exactly those 9, at the SAME
# versions, in the lab dataset -- a retag, not an upgrade.
import re

running = 'cp' + str(sys.version_info.major) + str(sys.version_info.minor)

def dist_name(wheel_name):
    return wheel_name.split('-')[0].lower().replace('_', '-')

def locked_tag(wheel_name):
    """The cpXY this wheel is locked to, or None if it runs on any Python.

    abi3 and pure-python wheels are not locked; 'cp313-cp313' is.
    """
    if 'abi3' in wheel_name:
        return None
    m = re.search(r'-(cp3\d+)-(cp3\d+)', wheel_name)
    return m.group(1) if m else None

OVERLAY = LAB / 'vendor' / running
overlay = {dist_name(w.name): w for w in OVERLAY.glob('*.whl')} if OVERLAY.is_dir() else {}

PLAN = []        # wheels to install, as paths
UNRESOLVED = []  # version-locked with no substitute -- these are fatal
for w in sorted(WHEELHOUSE.glob('*.whl')):
    if 'cutlass' in w.name.lower():
        continue
    tag = locked_tag(w.name)
    if tag is None or tag == running:
        PLAN.append(w)
        continue
    sub = overlay.get(dist_name(w.name))
    if sub is not None:
        PLAN.append(sub)
    else:
        UNRESOLVED.append(w.name)

tags = set()
for w in WHEELHOUSE.glob('*.whl'):
    tags.update(re.findall(r'-(cp3\d+)-', w.name))
say('running ' + running + '; wheelhouse compiled tags ' + str(sorted(tags)))
say('overlay ' + str(OVERLAY) + ': ' + str(len(overlay)) + ' wheel(s)')
say('install plan: ' + str(len(PLAN)) + ' wheel(s), ' + str(len(UNRESOLVED)) + ' unresolved')
for n in UNRESOLVED:
    say('  UNRESOLVED ' + n)

assert not UNRESOLVED, (
    'These wheels are built only for another Python and have no vendored '
    + running + ' substitute: ' + str(UNRESOLVED) + '. Download them at the SAME '
    'versions into vendor/' + running + '/ and re-run scripts/push_dataset.py. '
    'Do NOT change versions: the wheelhouse pins are what the harness expects.'
)
say('every version-locked wheel is satisfiable on ' + running)

In [ ]:
# Gate: every arm's bundle must already be in the PUBLISHED lab dataset.
#
# Placed before the torch install deliberately. `push_dataset.py` is a separate
# publish action from `git commit`, so a bundle can exist locally, be committed,
# and still be absent here -- and arm 2 would then fail only AFTER arm 1 had
# spent ~3 GPU-h. The gate costs milliseconds and protects the allocation.
#
# `.yml` is checked for by name because it is NEVER discovered by the harness
# (only agent.yaml / root_agent.yaml), so a .yml-only bundle scores 0.00 while
# looking present.
BUNDLES = LAB / 'bundles'
_missing, _bad = [], []
for _arm in ['v10_clamp_timecap', 'v11_greedy']:
    _d = BUNDLES / _arm
    if not _d.is_dir():
        _missing.append(_arm)
        continue
    _roots = [q.name for q in _d.rglob('*') if q.name in ('agent.yaml', 'root_agent.yaml')]
    _ymls = [str(q.relative_to(_d)) for q in _d.rglob('agent.yml')]
    say('  ' + _arm.ljust(22) + ' roots=' + str(sorted(set(_roots))) + (' WARN .yml: ' + str(_ymls) if _ymls else ''))
    if not _roots:
        _bad.append(_arm + ' (no agent.yaml/root_agent.yaml)')
assert not _missing, (
    'bundle(s) absent from the published lab dataset: ' + str(_missing)
    + ' -- run: python scripts/push_dataset.py -m "<note>" and wait for the new version')
assert not _bad, 'bundle(s) present but undiscoverable: ' + str(_bad)
_runner = LAB / 'scripts' / 'session_runner.py'
assert _runner.is_file(), 'no session_runner.py at ' + str(_runner)
say('bundle gate OK for ' + str(len(['v10_clamp_timecap', 'v11_greedy'])) + ' arm(s)')

In [ ]:
# Gate: the task list must be IN the published dataset, and the right length.
_tl = LAB / 'tasks' / 'clean_dev.txt'
assert _tl.is_file(), (
    'no task list at ' + str(_tl) + ' -- push_dataset.py copies only a whitelist '
    "of task lists (scripts/push_dataset.py:53). Add 'clean_dev.txt' to it and "
    're-push the dataset.')
_ids = [x.strip() for x in _tl.read_text(encoding='utf-8').splitlines() if x.strip()]
say('task list clean_dev.txt: ' + str(len(_ids)) + ' id(s), expect 60')
assert len(_ids) == 60, (
    'task list has ' + str(len(_ids)) + ' ids, not 60. The dataset copy is '
    'stale or truncated; re-push it. Both arms must run the SAME set or the '
    'pairing is meaningless.')
assert len(set(_ids)) == len(_ids), 'duplicate ids in clean_dev.txt'

In [ ]:
# Put the torch the prebuilt vllm was COMPILED against on this image, before the
# wheelhouse install, so vllm/_C.abi3.so links.
#
# MEASURED: wheelhouse vllm-0.19.1 declares `Requires-Dist: torch==2.10.0` and
# `Requires-Python: <3.14,>=3.10`. Kaggle's image ships torch 2.11.0+cu128, and
# vllm/_C then fails with `undefined symbol: _ZN3c1013MessageLoggerC1EPKciib`, a
# libtorch C++ symbol. abi3 pins the PYTHON abi, not the C++ one. So Python 3.13
# was never the blocker -- torch being one minor version off is.
#
# The wheels ride in from kernel 12 (CPU + internet), because torch is not in the
# wheelhouse and this kernel has no internet.
# All THREE of torch, torchvision and torchaudio, because they are one unit.
# Installing torch alone left the image's torchvision (built for 2.11) in place and
# the server died with `RuntimeError: operator torchvision::nms does not exist` --
# torchvision registers its compiled ops into torch's dispatcher, so a mismatched
# pair fails at registration rather than at import. vllm-0.19.1 pins exactly
# torch==2.10.0, torchvision==0.25.0, torchaudio==2.10.0.
TORCH_WANT = '2.10.0'
WANT = (('torch', '2.10.0'), ('torchvision', '0.25.0'), ('torchaudio', '2.10.0'))

cands = sorted(Path('/kaggle/input').rglob('torch-' + TORCH_WANT + '*.whl'))
say('torch ' + TORCH_WANT + ' wheels found: ' + str([c.name for c in cands]))
assert cands, (
    'no torch ' + TORCH_WANT + ' wheel under /kaggle/input. Run kernel '
    '12_fetch_torch and attach it via kernel_sources. Without it vllm cannot link.')

TORCH_DIR = cands[0].parent
PIN_WHEELS = []
for pkg, ver in WANT:
    hit = sorted(TORCH_DIR.glob(pkg + '-' + ver + '*.whl'))
    assert hit, (
        'no ' + pkg + ' ' + ver + ' wheel in ' + str(TORCH_DIR) + '. torch, '
        'torchvision and torchaudio must be installed as a matched set, or '
        'torchvision fails to register its ops against torch.')
    PIN_WHEELS.append(str(hit[0]))
    say('  will install ' + hit[0].name)

say('installing the matched torch set from ' + str(TORCH_DIR))
# WITH deps, from that directory only: torch 2.10 and 2.11 pin different nvidia-*
# versions and the image's are installed for 2.11. Mixing them is the same class of
# mismatch being fixed here, so the matching set is installed rather than trusted.
# Install by WHEEL PATH, not by requirement. The wheel is torch-2.10.0+cu128-...,
# and a `torch==2.10.0` specifier is ambiguous enough to have already broken the
# fetch step ("Cannot install torch==2.10.0 and torch==2.10.0+cu128"). A path
# cannot be misresolved. --find-links still covers the nvidia-* dependencies.
r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall',
                    '--no-index', '--find-links', str(TORCH_DIR)] + PIN_WHEELS,
                   capture_output=True, text=True)
say('torch set pip rc ' + str(r.returncode))
if r.returncode != 0:
    say((r.stderr or '')[-2500:])

# Verify the ops actually REGISTER, not merely that the versions read correctly.
# `torch.ops.torchvision.nms` is the exact operator whose absence killed the
# previous run, so it is the honest check: a version string can agree while the
# dispatcher disagrees.
chk = subprocess.run([sys.executable, '-c',
                      'import torch, torchvision, torchaudio;'
                      'print("torch", torch.__version__, torch.version.cuda);'
                      'print("torchvision", torchvision.__version__);'
                      'print("torchaudio", torchaudio.__version__);'
                      'print("nms op", torch.ops.torchvision.nms is not None)'],
                     capture_output=True, text=True)
say('torch set now: ' + ((chk.stdout or '') + (chk.stderr or '')).strip()[-800:])
assert chk.returncode == 0 and TORCH_WANT in (chk.stdout or ''), (
    'the matched torch set is not active, or torchvision ops do not register. '
    'A mismatched torchvision raises "operator torchvision::nms does not exist" '
    'inside the inference server, long after this point.')

In [ ]:
for pat in ('/usr/local/lib/python*/dist-packages/*cutlass*.pth',
            '/usr/local/lib/python*/site-packages/*cutlass*.pth'):
    for f in glob.glob(pat):
        try:
            os.unlink(f)
        except OSError:
            pass

# Install the PLAN from the previous cell, not the raw wheelhouse: the plan has
# already substituted a vendored cp313 build for each wheel the wheelhouse ships
# only as cp312.
tmp = Path('/tmp/wheelhouse')
tmp.mkdir(parents=True, exist_ok=True)
for w in PLAN:
    name = w.name
    if 'cu128' in name and '+' not in name:
        name = name.replace('cu128', '+cu128')
    if not (tmp / name).exists():
        try:
            os.symlink(w, tmp / name)
        except OSError:
            shutil.copy(w, tmp / name)

wheels = sorted(str(w) for w in tmp.glob('*.whl'))
say('installing ' + str(len(wheels)) + ' wheels')

PIP = [sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '--force-reinstall']
r = subprocess.run(PIP + wheels, capture_output=True, text=True)
say('bulk pip rc ' + str(r.returncode))

if r.returncode != 0:
    # The official starter installs all wheels in ONE pip call, so a single
    # unsupported wheel aborts the batch and swegemma never lands. Seen on the T4
    # image: apache_tvm_ffi ... is not a supported wheel on this platform.
    say('bulk install failed; retrying per wheel')
    say((r.stderr or '')[-700:])
    failed = []
    for w in wheels:
        rr = subprocess.run(PIP + [w], capture_output=True, text=True)
        if rr.returncode != 0:
            failed.append(Path(w).name)
    say(str(len(failed)) + ' wheel(s) failed individually:')
    for f in failed:
        say('    ' + f)
    assert not failed, 'wheels failed to install: ' + str(failed)

# Prove the serving stack LINKS, not merely that pip exited 0 and the top-level
# package imports.
#
# `import vllm` is NOT sufficient, and assuming it was cost a full run on
# 2026-10-07: vllm/_C.abi3.so loads lazily, so `import vllm` printed 0.19.1
# happily and the inference server then died with
#   ImportError: vllm/_C.abi3.so: undefined symbol: _ZN3c1013MessageLoggerC1EPKciib
# That is a libtorch C++ symbol. `abi3` pins the PYTHON ABI, not the C++ one, so a
# vllm built against one torch does not link against another -- and torch is not
# in the wheelhouse, it comes from the image, which Kaggle changed underneath us.
#
# `import vllm._C` forces the native library to load, which is the cheapest check
# that reaches the real failure: seconds, against the ~2.5 minutes of model load
# and ~1 minute per task that the old check let us waste.
chk = subprocess.run(
    [sys.executable, '-c',
     'import torch;print("torch", torch.__version__, "cuda", torch.cuda.is_available());'
     'import vllm;print("vllm", vllm.__version__);'
     'import vllm._C;print("vllm._C linked OK")'],
    capture_output=True, text=True)
say('serving stack: ' + ((chk.stdout or '') + (chk.stderr or '')).strip()[-1200:])
assert chk.returncode == 0, (
    'the serving stack does not link on this image. If the error names an '
    'undefined symbol in vllm/_C.abi3.so, the wheelhouse vllm was built against a '
    'different torch than this image ships -- a C++ ABI mismatch that no cp-tag '
    'substitution can fix. See LESSONS.md 2026-10-07.')

pre = subprocess.run([sys.executable, str(LAB / 'scripts' / 'calibrate.py'), '--preflight'],
                     capture_output=True, text=True)
say('preflight:')
say((pre.stdout or '') + (pre.stderr or ''))
assert pre.returncode == 0, 'preflight failed - harness imports are broken'

In [ ]:
assert 'MAX_MODEL_LEN' in dir() or 'MAX_MODEL_LEN' in globals(), (
    'MAX_MODEL_LEN is not defined; the reused setup cells changed')
say('MAX_MODEL_LEN=' + str(MAX_MODEL_LEN))

In [ ]:
# The reused setup cells do not import importlib; this cell needs both it and
# importlib.util, and `import importlib` alone does not bind the submodule.
import importlib
import importlib.util

# session_runner refuses to run (exit 4) unless inline_snapshot and dirty_equals
# are importable. That gate is right: probe_c12 lost 5 of 12 tasks to pytest
# COLLECTION errors from exactly these two packages, which scores as unresolved
# and is indistinguishable from a bad patch. C004 measured them moving clean-N
# from 71 to 102.
#
# The first attempt at this kernel passed neither the gate nor a staging step, so
# both arms died ~30 s in, after creating an empty results directory. 0.76 GPU-h
# for two empty directory entries.
#
# scripts/stage_test_deps.py cannot be used here: it fetches from PyPI, and this
# competition refuses internet with the L4 accelerator. So the wheels travel in
# the lab dataset and are installed offline.
#
# --no-deps is deliberate and load-bearing. inline_snapshot declares
# pytest>=8.3.4 and rich>=13.7.1; letting pip satisfy those could upgrade
# pygments or rich in the interpreter the per-task venvs inherit, and a newer
# Pygments is already a KNOWN cause of benchmark test failures (rich asserts
# guess_lexer(...) == 'python' and newer Pygments answers 'ipython'). Changing
# the test environment to fix the test environment is how a measurement stops
# meaning anything. dirty_equals declares no required deps at all.
# vendor/ mirrors the repo subdirectory for subdirectory: test_deps here, and the
# cp313 wheel overlay (used by cell 3) alongside it.
VENDOR = LAB / 'vendor' / 'test_deps'
say('vendor/test_deps: ' + (str(len(list(VENDOR.glob('*.whl')))) + ' wheels'
                            if VENDOR.is_dir() else 'DIRECTORY MISSING'))

need = ['inline_snapshot', 'dirty_equals']
say('missing before staging: ' + str([m for m in need if importlib.util.find_spec(m) is None]))
assert VENDOR.is_dir(), 'no vendor/test_deps in the lab dataset; re-run scripts/push_dataset.py'


def wheel_for(module_name):
    """Find a vendored wheel providing this IMPORT name.

    Import name and distribution name differ in spelling (underscore vs dash), so
    both are tried. Nothing clever: these are leaf packages.
    """
    stem = module_name.lower()
    for cand in (stem, stem.replace('_', '-'), stem.replace('-', '_')):
        for w in sorted(VENDOR.glob('*.whl')):
            if w.name.lower().startswith(cand + '-'):
                return w
    return None


def pip_install(wheel):
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                        '--no-index', '--no-deps', '--find-links', str(VENDOR), str(wheel)],
                       capture_output=True, text=True)
    importlib.invalidate_caches()
    return r


# Install by following the actual ModuleNotFoundError chain rather than by
# predicting the dependency tree. The previous version installed inline_snapshot
# with --no-deps and assumed the image already had `executing`; it did not, and
# the run died after vLLM had already loaded.
#
# --no-deps plus install-only-what-is-missing is what keeps this safe. The hazard
# was never installing a package, it was UPGRADING one the per-task venvs inherit
# -- a newer Pygments is a known cause of benchmark test failures. A module that
# is absent cannot be downgraded by installing it, and one that is present is
# never requested, so neither pytest nor rich nor pygments can be touched here.
still = []
for m in need:
    for attempt in range(8):
        try:
            importlib.import_module(m)
            say('  import ' + m + ' OK')
            break
        except ModuleNotFoundError as exc:
            miss = getattr(exc, 'name', None) or str(exc)
            w = wheel_for(miss)
            if w is None:
                still.append(m + ' needs ' + str(miss) + ' (no vendored wheel)')
                say('  import ' + m + ' BLOCKED on ' + str(miss) + ': no wheel in vendor/test_deps')
                break
            say('  ' + m + ' needs ' + str(miss) + ' -> installing ' + w.name)
            rr = pip_install(w)
            if rr.returncode != 0:
                still.append(m + ': pip failed on ' + w.name)
                say((rr.stderr or '')[-800:])
                break
        except Exception as exc:
            still.append(m + ': ' + type(exc).__name__ + ': ' + str(exc))
            say('  import ' + m + ' FAILED: ' + type(exc).__name__ + ': ' + str(exc))
            break
    else:
        still.append(m + ': dependency chain deeper than 8')

# Fail HERE, before vLLM spends 2.5 minutes loading 26 GB of weights, rather than
# inside session_runner after the allocation is already committed.
assert not still, (
    'still cannot import ' + str(still) + '. session_runner would exit 4. '
    'Add the missing wheel to vendor/test_deps and re-push the dataset.')

# The harness must still import after touching the environment. This is the check
# whose absence let an earlier run reach a preflight failure instead of stopping.
#
# P003 (2026-10-12) proves this gate has to mirror session_runner's imports
# EXACTLY, not approximately. The old probe imported swegemma.evaluate.Evaluator,
# which still worked, so the gate passed -- and session_runner then died 14 s
# later on `from swegemma.models.discovery import validate_single_declared_model`,
# which the competition removed when it republished the wheelhouse on 2026-10-09.
# That symbol is GONE: the replacement is adk_submission.discover_declared_models
# (P004), and session_runner now enforces the exactly-one-model rule itself.
# A gate that checks a DIFFERENT symbol from the one the script needs is not a
# gate. So every import in start_server() is listed here, and the run stops in
# seconds instead of after a 4-minute torch install and a 26 GB model load.
#
# EXTENDED 2026-10-21 to cover build_config() as well, which was the same bug
# one level down. build_config() runs AFTER start_server(), i.e. after the 26 GB
# load, and it imports two GOOGLE ADK symbols the gate never checked --
# including `google.adk.apps._configs`, a PRIVATE module path. A rename there
# would have killed each arm ~25 minutes in, with the quota already spent, and
# the log would have blamed the harness rather than the import.
#
# Keep this list in sync with scripts/session_runner.py start_server() AND
# build_config(). `import torch` is listed for literal fidelity even though
# TORCH_CELL already deep-checks it.
probe = ('import pydantic, pydantic_core;'
         'import torch;'
         'from adk_submission import VllmConfig, VllmServer, discover_adapters,'
         ' discover_declared_models;'
         'from swegemma.config import ALLOWED_ADAPTER_EXTENSIONS;'
         'from google.adk.agents.context_cache_config import ContextCacheConfig;'
         'from google.adk.apps._configs import EventsCompactionConfig;'
         'from swegemma.config import EvalConfig, build_submission_limits;'
         'from swegemma.evaluate import Evaluator;'
         'print("harness ok", pydantic.VERSION, pydantic_core.__version__)')
h = subprocess.run([sys.executable, '-c', probe], capture_output=True, text=True)
say('harness check: ' + (h.stdout or h.stderr or '').strip()[:300])
if h.returncode != 0:
    # Name the API that moved, so the log says WHAT to fix rather than that
    # something is broken. Each import is retried alone to isolate the failure.
    for one in [x for x in probe.split(';') if x.strip().startswith(('import ', 'from '))]:
        r = subprocess.run([sys.executable, '-c', one], capture_output=True, text=True)
        say('  ' + ('ok  ' if r.returncode == 0 else 'FAIL') + ' ' + one.strip()[:110])
        if r.returncode != 0:
            say('       ' + (r.stderr or '').strip().splitlines()[-1][:200])
assert h.returncode == 0, (
    'the harness API session_runner needs is not importable; refusing to spend '
    'the allocation. The wheelhouse is NOT frozen -- it was republished '
    '2026-10-09 -- so check whether a symbol moved before assuming a bad install.')

In [ ]:
ARMS = ['v10_clamp_timecap', 'v11_greedy']
RUN_PREFIX = 'decode_'
TASK_LIST = 'clean_dev.txt'
MAX_TOOL_CALLS = 100
MAX_TIME_MINUTES = 30
CONCURRENCY = 12
# The per-task cap plus headroom for the model load and the final grade, so the
# subprocess timeout can never fire BEFORE the cap it is meant to backstop.
PER_ARM_TIMEOUT_S = 175 * 60

RUNNER = str(LAB / 'scripts' / 'session_runner.py')
WORK = Path('/kaggle/working')


def archive_everything(tag):
    """Persist after EVERY arm, so a failure in arm 2 cannot lose arm 1.

    This is the probe_c12 lesson: the run succeeded and the output did not come
    home. Archiving is cheap and a partial result is still a result.
    """
    made = []
    for d in ('results', 'ledger'):
        p = WORK / d
        if p.is_dir() and any(p.rglob('*')):
            base = str(WORK / ('eval_' + d))
            shutil.make_archive(base, 'zip', root_dir=p)
            made.append(Path(base + '.zip').name)
    say('[' + tag + '] archived: ' + (', '.join(made) or 'nothing yet'))


summary = []
for arm in ARMS:
    run_id = RUN_PREFIX + arm.split('_')[0]
    cmd = [sys.executable, RUNNER,
           '--lab', str(LAB), '--data', str(DATA), '--model-path', str(MODEL),
           '--variant', arm, '--task-list', TASK_LIST,
           '--run-id', run_id, '--concurrency', str(CONCURRENCY),
           '--max-time-minutes', str(MAX_TIME_MINUTES),
           '--max-tool-calls', str(MAX_TOOL_CALLS),
           '--max-model-len', str(MAX_MODEL_LEN)]
    say('=' * 70)
    say('ARM ' + arm + '  run_id=' + run_id)
    say('  ' + ' '.join(cmd))

    t0 = time.time()
    rc, out, err = -1, '', ''
    try:
        proc = subprocess.run(cmd, capture_output=True, text=True, timeout=PER_ARM_TIMEOUT_S)
        rc, out, err = proc.returncode, proc.stdout or '', proc.stderr or ''
    except subprocess.TimeoutExpired as exc:
        rc = -9
        out = exc.stdout if isinstance(exc.stdout, str) else (exc.stdout or b'').decode('utf-8', 'replace')
        err = exc.stderr if isinstance(exc.stderr, str) else (exc.stderr or b'').decode('utf-8', 'replace')
        err = err + NL + '*** KILLED at ' + str(PER_ARM_TIMEOUT_S // 60) + ' min ***'
    wall = time.time() - t0

    # Full stdout to a file per arm: the ledger is the record, but when a run
    # fails the stdout is the only thing that says why.
    (WORK / ('stdout_' + run_id + '.txt')).write_text(
        NL.join(['rc=' + str(rc), 'wall_s=' + format(wall, '.0f'), '',
                 '=== STDOUT ===', out, '', '=== STDERR ===', err]),
        encoding='utf-8')

    say('  rc=' + str(rc) + '  wall=' + format(wall / 60, '.1f') + ' min')

    # Reprint this arm's ledger rows immediately. An interactive session can be
    # lost at any moment and publishes nothing, so a row that has been printed is
    # banked and a row that has not is gone.
    rows_here = [ln for ln in out.splitlines() if ln.startswith('LEDGER_ROW ')]
    say('  ledger rows echoed by this arm: ' + str(len(rows_here)))
    for ln in rows_here:
        print(ln, flush=True)

    say('  --- stdout tail ---')
    say(out[-6000:] or '(empty)')
    if rc != 0:
        say('  --- stderr tail ---')
        say(err[-3000:] or '(empty)')
    summary.append((arm, run_id, rc, wall))
    archive_everything(arm)

say('=' * 70)
for arm, run_id, rc, wall in summary:
    say(arm.ljust(22) + ' rc=' + str(rc).rjust(3) + '  ' + format(wall / 60, '5.1f') + ' min')

In [ ]:
# Behavioural guard-rail, per arm, straight into the log.
#
# `reissue_rate` = repeat-argument tool calls / calls that actually spent budget.
# It is how this project detects an agent looping on one failing action: a
# freed budget once raised it by +0.151, with 174 `submit_patch` calls in one
# task and 136 malformed `edit_file` in another. Greedy decode is deterministic
# given the context, so it is exactly the intervention that could make a loop
# self-sustaining -- the validator warns about it by name.
#
# Wrapped: a reporting bug must never destroy a completed run.
try:
    sys.path.insert(0, str(LAB / 'scripts'))
    sys.path.insert(0, str(LAB / 'src'))
    import analyze_traces

    say('=== GUARD-RAIL: behavioural metrics per arm ===')
    _per_arm = {}
    for arm in ARMS:
        run_id = RUN_PREFIX + arm.split('_')[0]
        tdir = WORK / 'results' / run_id / 'traces'
        if not tdir.is_dir():
            say('  ' + run_id + ': no traces dir at ' + str(tdir))
            continue
        m = analyze_traces.arm_metrics(tdir)
        _per_arm[run_id] = m
        say('  ' + run_id + ': ' + str(len(m)) + ' trace(s)')
        for key in analyze_traces.COMPARED:
            vals = [v[key] for v in m.values()]
            if vals:
                vals_sorted = sorted(vals)
                say('    ' + key.ljust(18)
                    + ' mean ' + format(sum(vals) / len(vals), '.3f')
                    + '  median ' + format(vals_sorted[len(vals_sorted) // 2], '.3f')
                    + '  max ' + format(max(vals), '.3f'))
        # Name the worst offenders: a mean can hide one pathological task, and
        # one pathological task is what the kill criterion is written against.
        worst = sorted(m.items(), key=lambda kv: -kv[1]['reissue_rate'])[:3]
        for t, mm in worst:
            say('    top reissue  ' + t.ljust(18)
                + ' rate ' + format(mm['reissue_rate'], '.3f')
                + '  calls ' + format(mm['counted_calls'], '.0f'))

    # The paired delta, which is what the pre-registered reading asks for.
    if len(_per_arm) == 2:
        (ra, ma), (rb, mb) = list(_per_arm.items())
        shared = sorted(set(ma) & set(mb))
        say('  paired on ' + str(len(shared)) + ' task(s): ' + rb + ' minus ' + ra)
        for key in analyze_traces.COMPARED:
            if shared:
                d = [mb[t][key] - ma[t][key] for t in shared]
                say('    d_' + key.ljust(18) + format(sum(d) / len(d), '+.3f'))
        say('  KILL CRITERION: d_reissue_rate >= +0.100 refutes greedy decode.')
except Exception as exc:
    say('guard-rail cell raised (not fatal): ' + type(exc).__name__ + ': ' + str(exc))

In [ ]:
# Echo every ledger row to stdout. Wrapped, because a bug in reporting must
# never hide a successful run -- and the log is recoverable even if a zip is not
# (scripts/recover_from_log.py exists for exactly that).
try:
    for arm in ARMS:
        run_id = RUN_PREFIX + arm.split('_')[0]
        jl = WORK / 'ledger' / 'runs' / run_id / 'task_results.jsonl'
        say('=== LEDGER ' + run_id + ' ===')
        if not jl.is_file():
            say('  MISSING: ' + str(jl))
            continue
        rows = [json.loads(x) for x in jl.read_text(encoding='utf-8').splitlines() if x.strip()]
        say('  rows: ' + str(len(rows)))
        for r in rows:
            say('  ROW ' + json.dumps(r, sort_keys=True))
        n = len(rows) or 1
        say('  resolved  ' + str(sum(1 for r in rows if r.get('resolved'))) + '/' + str(len(rows)))
        say('  gold_hit  ' + str(sum(1 for r in rows if r.get('gold_hit'))) + '/' + str(len(rows)))
        calls = [r.get('tool_calls_used') or 0 for r in rows]
        say('  calls: median ' + str(sorted(calls)[len(calls) // 2]) + '  max ' + str(max(calls)))
        say('  wall:  mean ' + format(sum(r.get('wall_seconds') or 0 for r in rows) / n / 60, '.1f') + ' min')
except Exception as exc:
    say('report cell raised (not fatal): ' + type(exc).__name__ + ': ' + str(exc))

try:
    archive_everything('final')
except Exception as exc:
    say('archive raised (not fatal): ' + str(exc))

say('=== files in /kaggle/working ===')
for q in sorted(WORK.iterdir()):
    say('  ' + q.name + '  ' + (str(q.stat().st_size) if q.is_file() else '(dir)'))

# The copy block. An interactive session publishes nothing, so this printed text
# is the primary record -- not a convenience. Rebuilt from the ledger FILES
# rather than from captured stdout, so it is complete even if an arm's subprocess
# output was truncated. The LEDGER_ROW prefix is what recover_from_log.py parses,
# so it is reapplied here verbatim.
print()
print('=== COPY FROM HERE ===')
total = 0
for arm in ARMS:
    run_id = RUN_PREFIX + arm.split('_')[0]
    jl = WORK / 'ledger' / 'runs' / run_id / 'task_results.jsonl'
    if not jl.is_file():
        print('# MISSING ledger for ' + run_id)
        continue
    for line in jl.read_text(encoding='utf-8').splitlines():
        if line.strip():
            print('LEDGER_ROW ' + line.strip())
            total += 1
print('=== TO HERE ===')
print()
print('rows in block: ' + str(total) + '   (expect 120: 60 tasks x 2 arm(s))')
print('Save the block as ledger/incoming/decode_v11.log, then run:')
print('  python scripts/recover_from_log.py --log ledger/incoming/decode_v11.log --write')